In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')

In [3]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
df.columns

Index(['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors',
       'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'fuel_efficiency_mpg'],
      dtype='str')

In [5]:
len(df)

10000

In [6]:
df.fuel_type.nunique()

3

In [7]:
df.isnull().sum() 

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [8]:
(df.isnull().sum() > 0).sum()

np.int64(2)

In [9]:
df.isna().sum()  

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [10]:
df['origin'].value_counts().loc['Asia']

np.int64(2994)

In [11]:
df[df['origin'] == 'Asia']['fuel_efficiency_mpg'].max()

np.float64(41.2)

Breaking that down left to right:

df['origin'] == 'Asia' → a True/False for every row
df[...] → keeps only the rows where that's True (all the Asia cars)
['fuel_efficiency_mpg'] → pulls just that column from the filtered rows
.max() → the largest value in it

After value_counts(), groupby(), or similar, the result is a new, smaller object, it's not the same dataframe anymore, so you can't reach back into columns that summary no longer has. When you want to combine "filter by X" and "look at column Y," filter first, then select the column, on the original df.

In [12]:
df.horsepower

0       243.0
1       272.0
2       267.0
3       258.0
4       304.0
        ...  
9995    271.0
9996    244.0
9997    267.0
9998    270.0
9999    295.0
Name: horsepower, Length: 10000, dtype: float64

In [13]:
df['horsepower'].median()

np.float64(254.0)

In [14]:
df['horsepower'].mode()

0    252.0
Name: horsepower, dtype: float64

In [15]:
df['horsepower'].mode()[0]

np.float64(252.0)

In [16]:
df['horsepower'].median() 
mode_hp = df['horsepower'].mode()[0]
df['horsepower'] = df['horsepower'].fillna(mode_hp)
df['horsepower'].median() 

np.float64(252.0)

In [17]:
import numpy as np

asia = df[df['origin'] == 'Asia']
X = asia[['vehicle_weight', 'model_year']].head(7).values

XTX = X.T @ X
XTX_inv = np.linalg.inv(XTX)
y = np.array([1100, 1300, 800, 900, 1000, 1100, 1200])
w = XTX_inv @ X.T @ y

w.sum()

np.float64(0.36919696904925203)

1. XTX = X transpose times X
XTX = X.T @ X — X.T flips your 7×2 matrix into a 2×7 one, then @ (matrix multiplication) multiplies that by the original X. The result XTX is a small 2×2 matrix. Conceptually, this squashes your 7 data points down into a compact summary of how the two features (vehicle_weight, model_year) relate to each other and to themselves.

2. Invert XTX
XTX_inv = np.linalg.inv(XTX) — matrix inversion is the matrix equivalent of dividing by a number. Just like 5 * (1/5) = 1, a matrix times its inverse gives the identity matrix. This step is what makes it possible to "undo" XTX later and solve for the weights.

3. Create y
y = np.array([1100, 1300, 800, 900, 1000, 1100, 1200]) — these are your 7 target values, one per row of X (think: the actual price or outcome you're trying to predict for each of those 7 Asia cars).

4. Solve for w
w = XTX_inv @ X.T @ y — read right to left conceptually: X.T @ y relates your features back to the targets, then multiplying by XTX_inv solves for the best-fit weights. w ends up being a 2-element array — one weight per feature (vehicle_weight, model_year). This whole formula (XTX_inv @ X.T @ y) is literally the closed-form solution to linear regression, often written as w = (XᵀX)⁻¹Xᵀy — you just typed it out by hand instead of calling .fit().